
# Lesson 9: Checkpointing and Persistence in LangGraph

## Learning objectives
- Understand checkpointing and persistence.
- Learn the role of a checkpointer.
- Understand how thread IDs separate user sessions.
- Save graph state and retrieve it across invocations.

## Why checkpointing matters

Without checkpointing, a graph normally starts with the state supplied to each invocation.

Checkpointing allows LangGraph to save state during execution and retrieve it later, enabling persistent workflows and resumable conversations.

## Core components
- **State:** Data shared between graph nodes.
- **Checkpoint:** Saved snapshot of graph state.
- **Checkpointer:** Saves and loads checkpoints.
- **Thread ID:** Identifies the execution thread whose checkpoints should be retrieved.


In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

print("Checkpointer created successfully!")


Checkpointer created successfully!



## Building a persistent conversation

We will create a simple conversation graph.

The graph will:
1. Receive a user message.
2. Append it to conversation history.
3. Return the updated history.
4. Save the graph state through a checkpointer.

We use `operator.add` as a reducer so new history entries are appended instead of replacing previous entries.

A consistent thread ID allows multiple invocations to access the same saved conversation.


In [2]:
from typing import Annotated
import operator

class ConversationState(TypedDict):
    user_message: str
    history: Annotated[list[str], operator.add]
    response: str


def conversation_node(state: ConversationState):
    message = state["user_message"]

    print("Processing:", message)

    return {
        "history": [message],
        "response": f"I received your message: {message}"
    }

In [3]:
builder = StateGraph(ConversationState)

builder.add_node("conversation", conversation_node)

builder.add_edge(START, "conversation")
builder.add_edge("conversation", END)

graph = builder.compile(checkpointer=memory)

print("Persistent conversation graph created!")

Persistent conversation graph created!


In [4]:
config = {
    "configurable": {
        "thread_id": "user_101"
    }
}

result = graph.invoke(
    {"user_message": "My name is Adarsh."},
    config=config
)

print("History:", result["history"])
print("Response:", result["response"])

Processing: My name is Adarsh.
History: ['My name is Adarsh.']
Response: I received your message: My name is Adarsh.


In [5]:
result = graph.invoke(
    {"user_message": "I am learning LangGraph."},
    config=config
)

print("History:", result["history"])
print("Response:", result["response"])

Processing: I am learning LangGraph.
History: ['My name is Adarsh.', 'I am learning LangGraph.']
Response: I received your message: I am learning LangGraph.


In [6]:
result = graph.invoke(
    {"user_message": "I am also learning Agentic AI."},
    config=config
)

print(result["history"])

Processing: I am also learning Agentic AI.
['My name is Adarsh.', 'I am learning LangGraph.', 'I am also learning Agentic AI.']


In [7]:
checkpoint = graph.get_state(config)

print("Saved values:")
print(checkpoint.values)

print("\nCheckpoint metadata:")
print(checkpoint.metadata)

Saved values:
{'user_message': 'I am also learning Agentic AI.', 'history': ['My name is Adarsh.', 'I am learning LangGraph.', 'I am also learning Agentic AI.'], 'response': 'I received your message: I am also learning Agentic AI.'}

Checkpoint metadata:
{'source': 'loop', 'step': 7, 'parents': {}}



## Results and observations

We tested two conversation threads:

- `user_101`: Retained its earlier messages across invocations.
- `user_202`: Started with an independent conversation history.

We also inspected the saved state using `graph.get_state(config)`.

### Important limitation

`InMemorySaver` stores checkpoints only in process memory. It is suitable for experiments, but not reliable permanent storage.

For production applications, database-backed checkpointers such as SQLite or PostgreSQL can provide persistence beyond the lifetime of a Python process.

### Key takeaways
1. A checkpointer enables graph state persistence.
2. A thread ID identifies which conversation to resume.
3. Different thread IDs maintain independent states.
4. `graph.get_state(config)` retrieves saved state.
5. In-memory persistence is not the same as durable database persistence.

### Interview question

**Q: How does LangGraph maintain conversation state across multiple invocations?**

It uses a checkpointer to save graph state as checkpoints. A consistent thread ID allows subsequent invocations to retrieve and continue from the saved state.
